# Section 5.4 - Backtest Dynamics

This notebook builds the paper-ready figures for the backtest dynamics section.

It answers:
1. How do portfolios evolve over time?
2. Which methods suffer deeper drawdowns?
3. Which methods recover faster after losses?
4. Are the paths consistent with the aggregate performance results from Section 5.2?
5. Do the methods separate more clearly in specific market periods?

The section is intentionally minimal:
- `fig:cum_returns`
- `fig:drawdown`


In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd()
OUTPUT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "data_exports" / "final_experimental_setup",
    PROJECT_ROOT / "outputs" / "final_experimental_setup",
]
OUTPUT_DIR = next((path for path in OUTPUT_CANDIDATES if path.exists()), OUTPUT_CANDIDATES[0])
TABLE_DIR = OUTPUT_DIR / "tables"
RUNS_DIR = OUTPUT_DIR / "framework_runs"
FIGURE_DIR = OUTPUT_DIR / "paper_figures" / "section_5_4"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using output directory: {OUTPUT_DIR}")
print(f"Figures will be written to: {FIGURE_DIR}")

bt = pd.read_csv(TABLE_DIR / "backtest_summary.csv")
print(f"Backtest rows: {len(bt)}")


Using output directory: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup
Figures will be written to: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_4
Backtest rows: 135


In [2]:
METHOD_ORDER = [
    "equal_weight",
    "naive_risk_parity",
    "markowitz",
    "hrp_recursive",
    "hca_deprado_ew_nrp",
]

PALETTE = {
    "equal_weight": "#4C78A8",
    "naive_risk_parity": "#72B7B2",
    "markowitz": "#F58518",
    "hrp_recursive": "#54A24B",
    "hca_deprado_ew_nrp": "#E45756",
}


def method_label(method: str) -> str:
    labels = {
        "equal_weight": "Equal Weight (1/N)",
        "naive_risk_parity": "Naive Risk Parity",
        "markowitz": "Markowitz",
        "hrp_recursive": "HRP Recursive",
        "hca_deprado_ew_nrp": "HCA De Prado EW/NRP",
    }
    return labels.get(method, method)


def apply_paper_layout(fig: go.Figure, *, title: str, width: int = 1150, height: int = 650) -> None:
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        font=dict(size=13),
        margin=dict(l=70, r=40, t=90, b=65),
        legend=dict(orientation="h", x=0, y=1.02, xanchor="left", yanchor="bottom"),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )


def save_figure(fig: go.Figure, filename: str) -> None:
    path = FIGURE_DIR / filename
    pio.write_html(fig, file=path, include_plotlyjs="cdn", full_html=True)
    print(f"Saved: {path}")


## Representative Run

The representative run is chosen to be informative for the paper: a case where `Markowitz` underperforms clearly relative to the simpler alternatives, so the path differences are visually meaningful rather than arbitrary.


In [3]:
run_rows = []
for run_id, group in bt.groupby("run_id"):
    mw_row = group[group["method"] == "markowitz"]
    if mw_row.empty:
        continue
    mw_row = mw_row.iloc[0]
    best_row = group.loc[group["sharpe_ratio"].idxmax()]
    run_rows.append(
        {
            "run_id": run_id,
            "markowitz_sharpe": mw_row["sharpe_ratio"],
            "best_method": best_row["method"],
            "best_sharpe": best_row["sharpe_ratio"],
            "gap_to_best": best_row["sharpe_ratio"] - mw_row["sharpe_ratio"],
        }
    )

run_quality = pd.DataFrame(run_rows).sort_values(
    ["gap_to_best", "markowitz_sharpe"], ascending=[False, True]
)
RUN_ID = run_quality.iloc[0]["run_id"]
print("Selected representative run:", RUN_ID)
run_quality.head(10)


Selected representative run: multi_asset_etf_20200601_24m


,run_id,markowitz_sharpe,best_method,best_sharpe,gap_to_best
13,multi_asset_etf_20200601_24m,0.193524,equal_weight,2.546559,2.353035
14,multi_asset_etf_20200601_36m,0.401885,equal_weight,2.546559,2.144674
12,multi_asset_etf_20200601_12m,0.449346,equal_weight,2.546559,2.097213
5,djia_20200601_36m,1.536411,equal_weight,2.496653,0.960242
4,djia_20200601_24m,1.620199,equal_weight,2.496653,0.876454
23,sp100_style_top100_20200601_36m,1.896605,equal_weight,2.738005,0.841400
26,sp100_style_top100_20220103_36m,-0.957516,naive_risk_parity,-0.362061,0.595455
25,sp100_style_top100_20220103_24m,-0.940850,naive_risk_parity,-0.367365,0.573484
8,djia_20220103_36m,-0.894429,hca_deprado_ew_nrp,-0.344017,0.550412
7,djia_20220103_24m,-0.844620,hca_deprado_ew_nrp,-0.319377,0.525243


## Figure 1 - Cumulative Returns

This figure answers: how do the portfolios evolve over time, and do the paths confirm the aggregate ranking from Section 5.2?


In [4]:
fig_cum = go.Figure()

for method in METHOD_ORDER:
    path = RUNS_DIR / RUN_ID / "backtests" / method / "cumulative_returns.csv"
    if not path.exists():
        continue
    series = pd.read_csv(path, index_col=0)
    series.index = pd.to_datetime(series.index)
    fig_cum.add_trace(
        go.Scatter(
            x=series.index,
            y=series.iloc[:, 0],
            mode="lines",
            name=method_label(method),
            line=dict(color=PALETTE[method], width=2.8),
        )
    )

fig_cum.update_xaxes(title="Date")
fig_cum.update_yaxes(title="Cumulative wealth (start = 1.0)")
apply_paper_layout(fig_cum, title=f"Cumulative Return Paths - {RUN_ID}")
save_figure(fig_cum, "figure_01_cumulative_returns.html")
fig_cum


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_4\figure_01_cumulative_returns.html


## Figure 2 - Drawdown Series

This figure answers: which methods crash harder, and which ones recover more smoothly after losses?


In [5]:
fig_dd = go.Figure()

for method in METHOD_ORDER:
    path = RUNS_DIR / RUN_ID / "backtests" / method / "drawdown_series.csv"
    if not path.exists():
        continue
    series = pd.read_csv(path, index_col=0)
    series.index = pd.to_datetime(series.index)
    fig_dd.add_trace(
        go.Scatter(
            x=series.index,
            y=series.iloc[:, 0],
            mode="lines",
            name=method_label(method),
            line=dict(color=PALETTE[method], width=2.8),
        )
    )

fig_dd.update_xaxes(title="Date")
fig_dd.update_yaxes(title="Drawdown")
apply_paper_layout(fig_dd, title=f"Drawdown Paths - {RUN_ID}")
save_figure(fig_dd, "figure_02_drawdown_series.html")
fig_dd


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_4\figure_02_drawdown_series.html
